In [2]:
import pandas as pd

df = pd.read_csv(r'../../datasets/CA_with_labels.csv')
df.head()

#print(df[df["name"] == "APT."])

# print(df['popularity'].value_counts())
# df['popularity'].value_counts().plot(kind='bar')
# plt.title('Popularity Distribution')

,spotify_id,name,artists,daily_rank,daily_movement,weekly_movement,country,snapshot_date,popularity,is_explicit,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,average_song
0,2CGNAOSuO1MEFCbBRgUzjd,luther (with sza),"Kendrick Lamar, SZA",1,0,2,CA,2025-02-17,90,False,...,-7.546,1,0.1250,0.2510,0.000000,0.2480,0.576,138.008,4,About_Average
1,6AI3ezQ4o3HUoP6Dhudph3,Not Like Us,Kendrick Lamar,2,0,3,CA,2025-02-17,92,True,...,-7.001,1,0.0776,0.0107,0.000000,0.1410,0.214,101.061,4,Higher
2,3GCdLUSnKSMJhs4Tj6CV3s,All The Stars (with SZA),"Kendrick Lamar, SZA",3,1,19,CA,2025-02-17,90,True,...,-4.946,1,0.0599,0.0612,0.000195,0.0926,0.557,96.782,4,About_Average
3,2plbrEY59IikOBgBGLjaoe,Die With A Smile,"Lady Gaga, Bruno Mars",4,2,-3,CA,2025-02-17,98,False,...,-7.777,0,0.0304,0.3080,0.000000,0.1220,0.535,157.969,3,Lower
4,0aB0v4027ukVziUGwVGYpG,tv off (feat. lefty gunplay),"Kendrick Lamar, Lefty Gunplay",5,0,5,CA,2025-02-17,92,True,...,-6.679,0,0.2630,0.0837,0.000000,0.4230,0.548,100.036,4,Higher


In [3]:

from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import RidgeClassifier

#a = df[df["name"] == "APT."]
X = df.drop(columns=["spotify_id", "name", "artists", "snapshot_date", "country", "album_name", "album_release_date", "popularity"], axis=1, inplace=False)
y = df["popularity"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.columns)

scaler = MinMaxScaler()
encode = OneHotEncoder()


Index(['daily_rank', 'daily_movement', 'weekly_movement', 'is_explicit',
       'duration_ms', 'danceability', 'energy', 'key', 'loudness', 'mode',
       'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'time_signature', 'average_song'],
      dtype='object')


In [4]:
from sklearn.compose import make_column_transformer
from sklearn.metrics import mean_squared_error


preprocessing = make_column_transformer((encode, ['average_song']), (scaler, ['key', 'daily_rank', 'daily_movement', 'weekly_movement',
       'is_explicit', 'duration_ms', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'time_signature'] ), remainder='passthrough')

pipeline = make_pipeline(
    preprocessing,
    RidgeClassifier()
)
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)


print("Ridge Regression")
mean_squared_error(y_test, y_pred)


Ridge Regression


239.135266723116

In [5]:
pred = pd.DataFrame(y_pred).value_counts()
test = pd.DataFrame(y_test).value_counts()

print(pred.describe())
print(test.describe())

count     17.000000
mean     277.882353
std      283.939889
min        2.000000
25%       27.000000
50%      169.000000
75%      529.000000
max      751.000000
Name: count, dtype: float64
count     74.000000
mean      63.837838
std      102.629674
min        1.000000
25%        2.000000
50%        6.500000
75%       82.500000
max      345.000000
Name: count, dtype: float64


In [8]:
hyperParameters = {'ridgeclassifier__alpha':[0.1, 0.5, 1, 5, 10, 50, 100, 500, 1000, 5000, 10000, 50000, 100000, 500000, 1000000]}

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold

RRGrid = GridSearchCV(
    pipeline,
    param_grid=hyperParameters,
    scoring="neg_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    verbose=3,
)

RRGrid.fit(X_train, y_train)
print("Best alpha: ", RRGrid.best_params_["ridgeclassifier__alpha"])
print("Best score: ", RRGrid.best_score_)

Fitting 5 folds for each of 15 candidates, totalling 75 fits
[CV 1/5] END .....ridgeclassifier__alpha=0.1;, score=-242.120 total time=   0.0s
[CV 2/5] END .....ridgeclassifier__alpha=0.1;, score=-213.254 total time=   0.0s
[CV 3/5] END .....ridgeclassifier__alpha=0.1;, score=-208.723 total time=   0.0s
[CV 4/5] END .....ridgeclassifier__alpha=0.1;, score=-221.359 total time=   0.0s
[CV 5/5] END .....ridgeclassifier__alpha=0.1;, score=-215.713 total time=   0.0s
[CV 1/5] END .....ridgeclassifier__alpha=0.5;, score=-242.164 total time=   0.0s
[CV 2/5] END .....ridgeclassifier__alpha=0.5;, score=-215.481 total time=   0.0s
[CV 3/5] END .....ridgeclassifier__alpha=0.5;, score=-208.752 total time=   0.0s
[CV 4/5] END .....ridgeclassifier__alpha=0.5;, score=-221.634 total time=   0.0s
[CV 5/5] END .....ridgeclassifier__alpha=0.5;, score=-213.821 total time=   0.0s
[CV 1/5] END .......ridgeclassifier__alpha=1;, score=-240.358 total time=   0.0s
[CV 2/5] END .......ridgeclassifier__alpha=1;, s